# 08 · El pipeline completo y cierre

**Objetivo:** ver el proyecto entero en pocas líneas, comprobar la trazabilidad entre capas y repasar qué cumple cada requisito.

In [0]:
%run ./00_Setup

## Parte 1 · El pipeline en seis líneas
Lo mismo que hace el Job, cada capa procesa **solo lo nuevo**.

In [0]:
download_days(date.today() - timedelta(days=4), 3)   # RAW:    API → JSON en el Volumen (idempotente)
run_bronze()                                          # BRONZE: Auto Loader lee solo los archivos nuevos
build_silver()                                        # SILVER: MERGE con los cambios de bronze (CDF)
validate_silver(1000)                                 # CALIDAD: si falla, no se publica
build_gold()                                          # GOLD:   dims + fact con los cambios de silver (CDF)
log_run("OK", "Cierre manual")                        # LOG

## Parte 2 · Trazabilidad entre capas
Cada vuelo que se pierde entre capas tiene una razón documentada.

In [0]:
b = spark.table(BRONZE)
traza = [
    ("Registros en RAW",                  spark.read.option("multiLine", "true").schema(ESQUEMA_VUELOS).json(RAW_PATH).count()),
    ("Vuelos únicos en bronze",           b.count()),
    ("  − sin hora de salida",            b.where("firstSeen IS NULL").count()),
    ("  − duración > 20 h",               b.where("lastSeen - firstSeen > 72000").count()),
    ("Vuelos en silver",                  spark.table(SILVER).count()),
    ("Vuelos en gold (fact)",             spark.table(FACT).count()),
]
for etapa, n in traza:
    print(f"{etapa:<30} {n:>8}")

In [0]:
display(spark.sql(f"SELECT path, commit_time FROM cloud_files_state('{CHECKPOINT_PATH}') ORDER BY commit_time DESC LIMIT 5"))
display(spark.sql(f"SELECT _commit_version, _change_type, count(*) AS filas FROM table_changes('{FACT}', 0) GROUP BY 1, 2 ORDER BY 1, 2"))
display(spark.table(LOG).orderBy(F.col("ejecutado_en").desc()))

## Parte 3 · Requisitos del proyecto

| Requisito | Dónde se cumple |
|---|---|
| Fuente: API pública | OpenSky `/flights/arrival` y `/departure` (notebook 01) |
| Fuente adicional: dataset público | OurAirports `airports.csv` + `countries.csv` (notebook 01) |
| RAW en volumen, sin transformar | `/Volumes/.../raw/flights` y `raw/catalogos` |
| Bronze en Delta con **Auto Loader** | `run_bronze()` (notebook 02) |
| Silver con **CDF** | `build_silver()` (notebook 03) |
| Gold con **CDF** y **modelo estrella** | `gold_fact_vuelo` + 3 dimensiones (notebook 04) |
| Job **calendarizado** | `proyecto_opensky_brasil`, diario 08:00 (notebook 05) |
| Logs y notificaciones (opcional) | `pipeline_log` + correo si falla |
| Time travel y mantenimiento | notebook 06 |
| Gobierno con Unity Catalog | notebook 07 |
| Reporte en Power BI | conectado a las tablas gold |

## Parte 4 · Problemas encontrados y cómo se resolvieron

| Problema | Solución |
|---|---|
| Poca cobertura de OpenSky en Perú | Se eligieron aeropuertos de Brasil (diagnóstico previo) |
| Límite de la API (429) | La descarga se detiene sola, guarda lo avanzado y es idempotente |
| Un vuelo aparece en dos aeropuertos | `MERGE` por `(icao24, firstSeen)` en bronze |
| Vuelos sin hora de salida | Se conservan en bronze, se excluyen en silver con regla documentada |
| Vuelos de más de 20 h | Regla de calidad en silver y en la validación del Job |
| 30 % de origen/destino desconocido | Limitación de la fuente; se marca `DESC` y se aclara en el reporte |

## ✔ Cierre
- [ ] El pipeline corre de punta a punta y cada capa procesa solo lo nuevo
- [ ] La trazabilidad RAW → bronze → silver → gold cuadra
- [ ] El Job está programado y deja registro en `pipeline_log`
- [ ] Gold está listo para Power BI